<a href="https://colab.research.google.com/github/tzs69/BT4012-kaggle-comp/blob/main/BT4012_competition_GNN_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph Models
Notebook to explore GNN models and compare them individually against stronger traditional models fitted in `BT4012_competition_baseline_modelling.ipynb`

In [ ]:
pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
# from google.colab import drive

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import GCNConv, SAGEConv
from sklearn.metrics import roc_auc_score

## Load Data

In [ ]:
# drive.mount('/content/drive')
# ROOT_DIR = "/content/drive/MyDrive/BT4012_competition"

ROOT_DIR = "."

for folder in [
    "transformed_datasets",
    "submissions",
]:
    Path(ROOT_DIR + f"/{folder}").mkdir(parents=True, exist_ok=True)

df_train = pd.read_csv(ROOT_DIR + "/transformed_datasets/train_topo.csv")
df_test = pd.read_csv(ROOT_DIR + "/transformed_datasets/test_topo.csv")
sample_submission = pd.read_csv(ROOT_DIR + "/base_datasets/sample_submission.csv")
txs_edgelist = pd.read_csv(ROOT_DIR + "/base_datasets/txs_edgelist.csv")

labeled = df_train.loc[df_train["label"].notna()].reset_index(drop=True)

y = labeled["label"].astype(int)

# Use only base feature rows for GNN models; doubtful that my self-engineered features will add much value
feature_cols = [c for c in labeled.columns if c.startswith("feat_")]
df_train = labeled[feature_cols + ["txId", "time_step", "label"]]
df_test = df_test[feature_cols + ["txId", "time_step"]]

assert labeled["txId"].is_unique
assert df_test["txId"].is_unique

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data pre-processing

Need to conduct a series of pre-processing steps to make the existing data at hand suitable for input into pytorch's PyG Graph-based models
- Merge train & test dfs
- Create txId to index mapping for merged df
- Map src and dest in edge list to the index mappings and filter to keep only tuples where both src & dest txns are represented in train or test data
- Split edge list into train/test


In [ ]:
df_combined = pd.concat([df_train, df_test], axis=0, ignore_index=True)

txId_to_idx = {txId: idx for idx, txId in enumerate(df_combined["txId"].to_numpy())}
idx_to_txId = {idx: txId for txId, idx in txId_to_idx.items()}

df_combined_for_graph = df_combined.copy()
df_combined_for_graph["map_index"] = df_combined_for_graph["txId"].map(txId_to_idx)
display(df_combined_for_graph[["txId", "map_index"]].head())
print("\n")

txs_edgelist_for_graph = txs_edgelist.copy()
txs_edgelist_for_graph["map_index_1"] = txs_edgelist_for_graph["txId1"].map(txId_to_idx)
txs_edgelist_for_graph["map_index_2"] = txs_edgelist_for_graph["txId2"].map(txId_to_idx)

# Filter out tuples where map_index_1 or map_index_2 or both is null
txs_edgelist_for_graph_cleaned = txs_edgelist_for_graph[
    txs_edgelist_for_graph["map_index_1"].notna() & txs_edgelist_for_graph["map_index_2"].notna()
].copy()

# Dropping all tuples where either src or dest (or both) txns do not exist inside labeled
# that all remaining edges fall within our training time period (1 <= time_step <= 35)
time_step_map = df_combined.set_index("txId")["time_step"]
txs_edgelist_for_graph_cleaned["time_step"] = txs_edgelist_for_graph_cleaned["txId1"].map(time_step_map)
assert txs_edgelist_for_graph_cleaned["time_step"].is_monotonic_increasing
assert txs_edgelist_for_graph_cleaned["time_step"].min() >= 1
assert txs_edgelist_for_graph_cleaned["time_step"].max() <= 49

display(txs_edgelist_for_graph_cleaned.head())
print(f"Num rows whr both src & dest nodes have known feature rows: {len(txs_edgelist_for_graph_cleaned)}")

,txId,map_index
0,232438397,0
1,232029206,1
2,232344069,2
3,27553029,3
4,3881097,4


,txId1,txId2,map_index_1,map_index_2,time_step
5,232344069,27553029,2.0,3.0,1
8,3881097,232457116,4.0,5.0,1
15,232051089,232470704,9.0,10.0,1
26,230473487,7089694,12.0,13.0,1
33,231182296,14660781,19.0,20.0,1


Num rows whr both src & dest nodes have known feature rows: 36624


### Edge index (using mapped indices instead of txIds) and txIds array for mapping

In [ ]:
txIds = torch.tensor(
    [idx_to_txId[i] for i in range(len(df_combined))],
    dtype=torch.int64
)
print(f"txIds shape: {txIds.shape}\n")


edge_index = torch.tensor(
    [
        txs_edgelist_for_graph_cleaned["map_index_1"].to_numpy(),
        txs_edgelist_for_graph_cleaned["map_index_2"].to_numpy()
    ],
    dtype=torch.int64
)
assert edge_index.min() >= 0
assert edge_index.max() < len(df_combined)
print(f"edge_index shape: {edge_index.shape}\n")

txIds shape: torch.Size([46564])

edge_index shape: torch.Size([2, 36624])



/tmp/ipykernel_931/587171982.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor(


### Train/Val/Test masks

In [ ]:
masks = [
    (("train_mask_1", range(1, 21)), ("val_mask_1", range(21, 26))), # fold, train_range, early_stop_range, val_range
    (("train_mask_2", range(1, 26)), ("val_mask_2", range(26, 31))),
    (("train_mask_3", range(1, 31)), ("val_mask_3", range(31, 36))),
]

train_masks = []
val_masks = []
train_mask_cols = []
val_mask_cols = []

for train_mask, val_mask in masks:
    df_combined[train_mask[0]] = df_combined["time_step"].between(
        train_mask[1][0],
        train_mask[1][-1]
    ) & df_combined["label"].notna()
    train_mask_cols.append(train_mask[0])

    df_combined[val_mask[0]] = df_combined["time_step"].between(
        val_mask[1][0],
        val_mask[1][-1]
    ) & df_combined["label"].notna()
    val_mask_cols.append(val_mask[0])

train_masks = torch.tensor(
    df_combined[train_mask_cols].to_numpy(),
    dtype=torch.bool
)
assert train_masks.sum(dim=0).tolist() == [18889, 24200, 26905]
print(f"train_masks shape: {train_masks.shape}\n")

val_masks = torch.tensor(
    df_combined[val_mask_cols].to_numpy(),
    dtype=torch.bool
)
assert val_masks.sum(dim=0).tolist() == [5311, 2705, 4330]
print(f"val_masks shape: {val_masks.shape}\n")

# Test timesteps: 36 - 49
df_combined["test_mask"] = df_combined["time_step"].between(36, 49)
test_mask = torch.tensor(
    df_combined["test_mask"].to_numpy(),
    dtype=torch.bool
)
assert test_mask.sum().item() == 15329
print(f"test_mask shape: {test_mask.shape}\n")


train_masks shape: torch.Size([46564, 3])

val_masks shape: torch.Size([46564, 3])

test_mask shape: torch.Size([46564])



### x & y
- For labels, impute nulls with -1

In [ ]:
y = torch.tensor(
    df_combined["label"].fillna(-1).to_numpy(),
    dtype=torch.float32
)
assert not torch.isnan(y).any()
assert torch.isin(y, torch.tensor([-1.0, 0.0, 1.0])).all()

# Ensure all train/val masked indices correspond to labelled rows
for i in range(3):
    assert torch.isin(y[train_masks[:, i]], torch.tensor([0.0, 1.0])).all()
    assert torch.isin(y[val_masks[:, i]], torch.tensor([0.0, 1.0])).all()

print(f"y shape: {y.shape}\n")

x = torch.tensor(
    df_combined[feature_cols].to_numpy(),
    dtype=torch.float32
)
assert torch.isfinite(x).all()
print(f"x shape: {x.shape}\n")

y shape: torch.Size([46564])

x shape: torch.Size([46564, 165])



### Combine all into PyG Data object

In [ ]:
data = Data(
    x=x,
    y=y,
    edge_index=edge_index,
    txIds=txIds,
    train_mask=train_masks,
    val_mask=val_masks,
    test_mask=test_mask
)

data.validate(raise_on_error=True)
print(data)

Data(x=[46564, 165], edge_index=[2, 36624], y=[46564], txIds=[46564], train_mask=[46564, 3], val_mask=[46564, 3], test_mask=[46564])


## Modular GNN Training / Testing Helper Functions

### K-fold CV

In [ ]:
def run_gnn_cv(
    MODEL_CLASS,
    data,
    INPUT_DIM,
    HIDDEN_DIM=32,
    DROPOUT=0.4,
    USE_UNDIRECTED=True,
    USE_POS_WEIGHT=False,
    K=3,
    MAX_EPOCHS=300,
    STALL_TOLERANCE=10,
    LEARNING_RATE=0.01,
    device=None
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Using model: {MODEL_CLASS.__name__}")
    print(f"Model Archi:")
    print(MODEL_CLASS(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, dropout=DROPOUT))

    cv_data = data.clone()
    if USE_UNDIRECTED:
        print("Using undirected edges")
        cv_data.edge_index = to_undirected(cv_data.edge_index, num_nodes=cv_data.num_nodes)
        print(f"edge_index shape: {cv_data.edge_index.shape}")
    cv_data = cv_data.to(device)

    oof_predictions = np.full(cv_data.num_nodes, np.nan, dtype=np.float32)
    fold_results = []

    print("\n")

    for fold in range(K):
        print(f"{24*"="} FOLD {fold+1} {24*"="}")

        torch.manual_seed(42)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42)

        model = MODEL_CLASS(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
        model.to(device)

        # Configure per-fold train data attributes
        fold_train_mask = cv_data.train_mask[:, fold]
        fold_val_mask = cv_data.val_mask[:, fold]
        target = cv_data.y[fold_train_mask]

        if USE_POS_WEIGHT:
            pos_weight = (target == 0).sum()/(target == 1).sum()
        else:
            pos_weight = torch.tensor(1.0, device=device)
        print("Positive weight:", pos_weight.item())

        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

        best_auc = -np.inf
        best_epoch = 0
        best_state = None
        epochs_without_improvement = 0

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            optimizer.zero_grad(set_to_none=True)

            logits = model(cv_data.x, cv_data.edge_index)
            loss = criterion(logits[fold_train_mask], target)

            loss.backward()
            optimizer.step()

            model.eval()

            with torch.no_grad():
                logits = model(cv_data.x, cv_data.edge_index)
                val_preds = torch.sigmoid(logits[fold_val_mask]).cpu().numpy()

                val_auc = roc_auc_score(
                    cv_data.y[fold_val_mask].cpu().numpy(),
                    val_preds
                )

            if val_auc > best_auc + 1e-5:
                best_auc = val_auc
                best_epoch = epoch

                best_state = {
                    name: value.detach().cpu().clone()
                    for name, value in model.state_dict().items()
                }

                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epoch == 1 or epoch % 10 == 0:
                print(
                    f"Epoch {epoch:03d} | "
                    f"loss={loss.item():.4f} | "
                    f"val_auc={val_auc:.5f} | "
                    f"best={best_auc:.5f}"
                )

            if epochs_without_improvement >= STALL_TOLERANCE:
                print(f"Early stopping; last epoch = {epoch}")
                break

        # Restore this fold's best model
        model.load_state_dict(best_state)
        model.eval()

        # Generate this fold's OOF predictions
        with torch.no_grad():
            best_logits = model(cv_data.x, cv_data.edge_index)
            best_val_preds = torch.sigmoid(best_logits[fold_val_mask]).cpu().numpy()

        oof_predictions[fold_val_mask.cpu().numpy()] = best_val_preds

        fold_results.append({
            "fold": fold + 1,
            "best_epoch": best_epoch,
            "best_auc": best_auc
        })

        print(f"Best epoch: {best_epoch}")
        print(f"Best Fold {fold + 1} AUC: {best_auc}\n")

    cv_results = pd.DataFrame(fold_results)

    median_best_epoch = int(np.median(cv_results["best_epoch"]))

    display(cv_results)
    print(f"Mean AUC: {cv_results['best_auc'].mean()}")
    print(f"AUC std: {cv_results['best_auc'].std()}")
    print(f"Median best epoch: {median_best_epoch}")

    return {
        "cv_results": cv_results,
        "median_best_epoch": median_best_epoch,
        "oof_predictions": oof_predictions
    }

### Train

In [ ]:
def run_gnn_train(
    MODEL_CLASS,
    data,
    MAX_EPOCHS,
    INPUT_DIM,
    HIDDEN_DIM=32,
    DROPOUT=0.4,
    USE_UNDIRECTED=False,
    USE_POS_WEIGHT=False,
    K=3,
    STALL_TOLERANCE=10,
    LEARNING_RATE=0.01,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    print(f"Using model: {MODEL_CLASS.__name__}")
    model = MODEL_CLASS(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
    model.to(device)

    train_data = data.clone()
    if USE_UNDIRECTED:
        print("Using undirected edges")
        train_data.edge_index = to_undirected(train_data.edge_index, num_nodes=train_data.num_nodes)
        print(f"edge_index shape: {train_data.edge_index.shape}")
    train_data = train_data.to(device)

    all_labelled_mask = train_data.y >= 0 # All labelled train rows
    target = train_data.y[all_labelled_mask]

    if USE_POS_WEIGHT:
        pos_weight = (target == 0).sum()/(target == 1).sum()
    else:
        pos_weight = torch.tensor(1.0, device=device)

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(train_data.x, train_data.edge_index)
        loss = criterion(logits[all_labelled_mask], train_data.y[all_labelled_mask])

        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch%10 == 0 or epoch == MAX_EPOCHS:
            print(
                f"Final epoch {epoch:03d}/{MAX_EPOCHS} | "
                f"loss={loss.item():.4f}"
            )

    # Move inference inputs to same device as trained model to safeguard testing
    return {
        "model": model,
        "test_x": train_data.x.to(device),
        "test_edge_index": train_data.edge_index.to(device),
        "test_mask": train_data.test_mask,
        "device": device
    }

### Test

In [ ]:
def run_gnn_test(
    model,
    test_x,
    test_edge_index,
    test_mask,
):
    """
    Returns:
    - sample_submission: pd.DataFrame in submission format
    """
    print(f"Using model: {model.__class__.__name__}")
    model.eval()
    print("\n")

    with torch.no_grad():
        logits = model(test_x, test_edge_index)
        test_preds = torch.sigmoid(logits)
        test_preds = test_preds[test_mask].cpu().numpy()

    assert len(test_preds) == len(df_test)
    assert np.isfinite(test_preds).all()
    assert np.all((test_preds >= 0) & (test_preds <= 1))

    # Copy sample submission and overlay sample "target" column w actual preds
    submission_df = sample_submission.copy()
    submission_df["target"] = test_preds

    display(submission_df.head())
    print("\n")
    display(submission_df["target"].describe())
    return submission_df

### Generate Run Name

In [ ]:
def get_run_name(model_name, USE_UNDIRECTED, USE_POS_WEIGHT):
    run_name = f"{model_name.lower()}"
    run_name += "_undirected" if USE_UNDIRECTED else "_directed"
    run_name += "_weighted" if USE_POS_WEIGHT else "_unweighted"
    return run_name

## GCN Baseline model

### Define GCN Model

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim, cached=True)
        self.conv2 = GCNConv(hidden_dim, 1, cached=True)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)

        return x.squeeze(-1)

### GCN K-fold CV

In [ ]:
# Hyperparameters and config settings (Use same for train/val)
GCN = GCN
GCN_USE_UNDIRECTED = True
GCN_USE_POS_WEIGHT = False
INPUT_DIM = data.num_node_features
HIDDEN_DIM = 32
DROPOUT = 0.4
K = 3
MAX_EPOCHS = 300
STALL_TOLERANCE = 10
LEARNING_RATE = 0.01
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


gcn_cv_results = run_gnn_cv(
    MODEL_CLASS=GCN,
    data=data,
    INPUT_DIM=INPUT_DIM,
    HIDDEN_DIM=HIDDEN_DIM,
    DROPOUT=DROPOUT,
    USE_UNDIRECTED=GCN_USE_UNDIRECTED,
    USE_POS_WEIGHT=GCN_USE_POS_WEIGHT,
    K=K,
    MAX_EPOCHS=MAX_EPOCHS,
    STALL_TOLERANCE=STALL_TOLERANCE,
    LEARNING_RATE=LEARNING_RATE,
    device=device
)

Using model: GCN
Model Archi:
GCN(
  (conv1): GCNConv(165, 32)
  (conv2): GCNConv(32, 1)
)
Using undirected edges
edge_index shape: torch.Size([2, 73248])


======================== FOLD 1 ========================
Positive weight: 1.0
Epoch 001 | loss=0.6781 | val_auc=0.82932 | best=0.82932
Epoch 010 | loss=0.1528 | val_auc=0.88379 | best=0.88379
Epoch 020 | loss=0.1305 | val_auc=0.88711 | best=0.88711
Epoch 030 | loss=0.1150 | val_auc=0.88619 | best=0.88773
Early stopping; last epoch = 33
Best epoch: 23
Best Fold 1 AUC: 0.8877309945004412

======================== FOLD 2 ========================
Positive weight: 1.0
Epoch 001 | loss=0.6627 | val_auc=0.72543 | best=0.72543
Epoch 010 | loss=0.1741 | val_auc=0.86146 | best=0.86146
Epoch 020 | loss=0.1503 | val_auc=0.90576 | best=0.90576
Epoch 030 | loss=0.1320 | val_auc=0.93481 | best=0.93481
Epoch 040 | loss=0.1212 | val_auc=0.94112 | best=0.94112
Epoch 050 | loss=0.1142 | val_auc=0.94713 | best=0.94713
Epoch 060 | loss=0.1075 | val_auc

,fold,best_epoch,best_auc
0,1,23,0.887731
1,2,247,0.972738
2,3,162,0.955615


Mean AUC: 0.9386945906495563
AUC std: 0.04495843614992946
Median best epoch: 162


### GCN Train

In [ ]:
gcn_cv_best_epoch = gcn_cv_results["median_best_epoch"]

gcn_train_out = run_gnn_train(
    MODEL_CLASS=GCN,
    data=data,
    INPUT_DIM=INPUT_DIM,
    HIDDEN_DIM=HIDDEN_DIM,
    DROPOUT=DROPOUT,
    MAX_EPOCHS=gcn_cv_best_epoch,
    USE_UNDIRECTED=GCN_USE_UNDIRECTED,
    USE_POS_WEIGHT=GCN_USE_POS_WEIGHT,
    LEARNING_RATE=LEARNING_RATE,
    device=device
)

Using model: GCN
Using undirected edges
edge_index shape: torch.Size([2, 73248])
Final epoch 001/162 | loss=0.6684
Final epoch 010/162 | loss=0.2065
Final epoch 020/162 | loss=0.1757
Final epoch 030/162 | loss=0.1551
Final epoch 040/162 | loss=0.1413
Final epoch 050/162 | loss=0.1328
Final epoch 060/162 | loss=0.1239
Final epoch 070/162 | loss=0.1187
Final epoch 080/162 | loss=0.1143
Final epoch 090/162 | loss=0.1103
Final epoch 100/162 | loss=0.1061
Final epoch 110/162 | loss=0.1029
Final epoch 120/162 | loss=0.1003
Final epoch 130/162 | loss=0.0983
Final epoch 140/162 | loss=0.0972
Final epoch 150/162 | loss=0.0973
Final epoch 160/162 | loss=0.0925
Final epoch 162/162 | loss=0.0944


### GCN Test Preds

In [ ]:
gcn_model = gcn_train_out["model"]
test_x = gcn_train_out["test_x"]
test_edge_index = gcn_train_out["test_edge_index"]
test_mask = gcn_train_out["test_mask"]

gcn_test_preds_df = run_gnn_test(
    model=gcn_model,
    test_x=test_x,
    test_edge_index=test_edge_index,
    test_mask=test_mask
)

Using model: GCN




,index,target
0,0,3.350652e-02
1,1,1.664728e-02
2,2,9.976051e-09
3,3,1.956828e-02
4,4,1.265391e-02


,target
count,15329.000000
mean,0.052012
std,0.158712
min,0.000000
25%,0.000026
50%,0.001901
75%,0.023305
max,0.998280


### GCN Submission Checkpoint
| Config | Score | Date | Time | Remarks |
|--------|-------|------|------|---------|
| directed & weighted | 0.88106 | 02/10/26 | 23:52 |
| undirected & unweighted | 0.88824 | 02/10/26 | 23:59 |

In [ ]:
gcn_run_name = get_run_name(
    model_name=gcn_model.__class__.__name__,
    USE_UNDIRECTED=GCN_USE_UNDIRECTED,
    USE_POS_WEIGHT=GCN_USE_POS_WEIGHT
)
gcn_submission_path = f"/content/drive/MyDrive/BT4012_competition/submissions/{gcn_run_name}.csv"
gcn_test_preds_df.to_csv(gcn_submission_path, index=False)

print(f"\nSaved to {gcn_submission_path}")


Saved to /content/drive/MyDrive/BT4012_competition/submissions/gcn_undirected_unweighted.csv


## GraphSAGE Baseline model

### Define GraphSAGE Model

In [ ]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim, aggr="mean")
        self.conv2 = SAGEConv(hidden_dim, 1, aggr="mean")
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)

        return x.squeeze(-1)

### GraphSAGE K-fold CV

In [ ]:
GSAGE = GraphSAGE
GSAGE_USE_UNDIRECTED = False
GSAGE_USE_POS_WEIGHT = False
INPUT_DIM = data.num_node_features
HIDDEN_DIM = 32
DROPOUT = 0.4
K = 3
MAX_EPOCHS = 300
STALL_TOLERANCE = 10
LEARNING_RATE = 0.01
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gsage_cv_results = run_gnn_cv(
    MODEL_CLASS=GSAGE,
    data=data,
    INPUT_DIM=INPUT_DIM,
    HIDDEN_DIM=HIDDEN_DIM,
    DROPOUT=DROPOUT,
    USE_UNDIRECTED=GSAGE_USE_UNDIRECTED,
    USE_POS_WEIGHT=GSAGE_USE_POS_WEIGHT,
    K=K,
    MAX_EPOCHS=MAX_EPOCHS,
    STALL_TOLERANCE=STALL_TOLERANCE,
    LEARNING_RATE=LEARNING_RATE,
    device=device
)

Using model: GraphSAGE
Model Archi:
GraphSAGE(
  (conv1): SAGEConv(165, 32, aggr=mean)
  (conv2): SAGEConv(32, 1, aggr=mean)
)


======================== FOLD 1 ========================
Positive weight: 1.0
Epoch 001 | loss=0.7844 | val_auc=0.82526 | best=0.82526
Epoch 010 | loss=0.1631 | val_auc=0.86985 | best=0.86985
Epoch 020 | loss=0.1315 | val_auc=0.87758 | best=0.87758
Epoch 030 | loss=0.1061 | val_auc=0.88107 | best=0.88107
Epoch 040 | loss=0.0930 | val_auc=0.88448 | best=0.88448
Epoch 050 | loss=0.0837 | val_auc=0.89023 | best=0.89023
Epoch 060 | loss=0.0768 | val_auc=0.89572 | best=0.89572
Epoch 070 | loss=0.0708 | val_auc=0.89950 | best=0.89950
Epoch 080 | loss=0.0662 | val_auc=0.90247 | best=0.90247
Epoch 090 | loss=0.0632 | val_auc=0.90592 | best=0.90592
Epoch 100 | loss=0.0592 | val_auc=0.90839 | best=0.90839
Epoch 110 | loss=0.0590 | val_auc=0.91093 | best=0.91093
Epoch 120 | loss=0.0555 | val_auc=0.91192 | best=0.91197
Early stopping; last epoch = 128
Best epoch: 118
Bes

,fold,best_epoch,best_auc
0,1,118,0.911972
1,2,116,0.976715
2,3,107,0.970878


Mean AUC: 0.9531881972446277
AUC std: 0.035813661448220194
Median best epoch: 116


### GraphSAGE Train

In [ ]:
gsage_cv_best_epoch = gsage_cv_results["median_best_epoch"]

gsage_train_out = run_gnn_train(
    MODEL_CLASS=GSAGE,
    data=data,
    INPUT_DIM=INPUT_DIM,
    HIDDEN_DIM=HIDDEN_DIM,
    DROPOUT=DROPOUT,
    MAX_EPOCHS=gsage_cv_best_epoch,
    USE_UNDIRECTED=GSAGE_USE_UNDIRECTED,
    USE_POS_WEIGHT=GSAGE_USE_POS_WEIGHT,
    LEARNING_RATE=LEARNING_RATE,
    device=device
)

Using model: GraphSAGE
Final epoch 001/116 | loss=0.7650
Final epoch 010/116 | loss=0.2081
Final epoch 020/116 | loss=0.1671
Final epoch 030/116 | loss=0.1388
Final epoch 040/116 | loss=0.1232
Final epoch 050/116 | loss=0.1122
Final epoch 060/116 | loss=0.1041
Final epoch 070/116 | loss=0.0953
Final epoch 080/116 | loss=0.0925
Final epoch 090/116 | loss=0.0881
Final epoch 100/116 | loss=0.0833
Final epoch 110/116 | loss=0.0819
Final epoch 116/116 | loss=0.0789


### GraphSAGE Test Preds

In [ ]:
gsage_model = gsage_train_out["model"]
test_x = gsage_train_out["test_x"]
test_edge_index = gsage_train_out["test_edge_index"]
test_mask = gsage_train_out["test_mask"]

gsage_test_preds_df = run_gnn_test(
    model=gsage_model,
    test_x=test_x,
    test_edge_index=test_edge_index,
    test_mask=test_mask
)

Using model: GraphSAGE




,index,target
0,0,0.004973
1,1,0.011490
2,2,0.002033
3,3,0.000021
4,4,0.059287


,target
count,15329.000000
mean,0.061551
std,0.179128
min,0.000000
25%,0.000001
50%,0.000618
75%,0.020109
max,1.000000


### GraphSAGE Submission Checkpoint
| Config | Score | Date | Time | Remarks |
|--------|-------|------|------|---------|
| undirected & unweighted | 0.91547 | 03/10/26 | 04:00 |
| directed & unweighted | 0.86079 | 03/10/26 | 16:15 | Error run: helper function bug
| directed & unweighted | 0.90796 | 03/10/26 | 17:30 |

In [ ]:
gsage_run_name = get_run_name(
    model_name=gsage_model.__class__.__name__,
    USE_UNDIRECTED=GSAGE_USE_UNDIRECTED,
    USE_POS_WEIGHT=GSAGE_USE_POS_WEIGHT
)
gsage_submission_path = f"/content/drive/MyDrive/BT4012_competition/submissions/{gsage_run_name}.csv"
gsage_test_preds_df.to_csv(gsage_submission_path, index=False)

print(f"\nSaved to {gsage_submission_path}")


Saved to /content/drive/MyDrive/BT4012_competition/submissions/graphsage_directed_unweighted.csv
